In [2]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader
from torchvision import datasets, transforms


# -------------------------
# 1. Config
# -------------------------
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
BATCH_SIZE = 128
LR = 1e-3
EPOCHS = 3


In [3]:


# -------------------------
# 2. Dataset
# -------------------------
transform = transforms.Compose([
    transforms.ToTensor(),
])

train_dataset = datasets.MNIST(
    root="./data",
    train=True,
    download=True,
    transform=transform,
)

test_dataset = datasets.MNIST(
    root="./data",
    train=False,
    download=True,
    transform=transform,
)

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
)

test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
)



In [4]:

# -------------------------
# 3. Models
# -------------------------
class Model1(nn.Module):
    """
    MNIST flattened input:
        [N, 784] -> [N, 100] -> [N, 40]
    """
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(784, 100),
            nn.ReLU(),
            nn.Linear(100, 40),
            nn.ReLU(),
        )

    def forward(self, x):
        x = x.view(x.size(0), -1)  # [N, 1, 28, 28] -> [N, 784]
        return self.net(x)


class Model2(nn.Module):
    """
    Boundary input:
        [N, 40] -> [N, 50] -> [N, 10]
    """
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(40, 50),
            nn.ReLU(),
            nn.Linear(50, 10),
        )

    def forward(self, h):
        return self.net(h)


model1 = Model1().to(DEVICE)
model2 = Model2().to(DEVICE)

optimizer1 = torch.optim.Adam(model1.parameters(), lr=LR)
optimizer2 = torch.optim.Adam(model2.parameters(), lr=LR)


# -------------------------
# 4. Train one epoch
# -------------------------
def train_one_epoch(epoch):
    model1.train()
    model2.train()

    total_loss = 0.0
    correct = 0
    total = 0

    for step, (x, y) in enumerate(train_loader):
        x = x.to(DEVICE)
        y = y.to(DEVICE)

        optimizer1.zero_grad()
        optimizer2.zero_grad()

        # -------------------------
        # Forward: Stage 1
        # -------------------------
        h = model1(x)  # [N, 40]

        # h는 upstream graph에 연결되어 있음
        # h_detached는 downstream용 새 leaf tensor
        h_detached = h.detach().requires_grad_(True)

        # retain_grad는 leaf tensor에는 사실 필수는 아니지만,
        # pipeline boundary gradient 확인용으로 명시
        h_detached.retain_grad()

        # -------------------------
        # Forward: Stage 2
        # -------------------------
        logits = model2(h_detached)  # [N, 10]
        loss = F.cross_entropy(logits, y)

        # -------------------------
        # Backward: Stage 2
        # -------------------------
        loss.backward()

        # 여기서 h_detached.grad == dLoss / dHidden
        grad_h = h_detached.grad

        # -------------------------
        # Backward: Stage 1
        # -------------------------
        h.backward(grad_h)

        # -------------------------
        # Optimizer step
        # -------------------------
        optimizer1.step()
        optimizer2.step()

        total_loss += loss.item() * x.size(0)

        pred = logits.argmax(dim=-1)
        correct += (pred == y).sum().item()
        total += y.size(0)

        if step % 100 == 0:
            print(
                f"Epoch {epoch} | Step {step} | "
                f"Loss {loss.item():.4f} | "
                f"Boundary grad norm {grad_h.norm().item():.4f}"
            )

    avg_loss = total_loss / total
    acc = correct / total

    print(f"[Train] Epoch {epoch} | Loss {avg_loss:.4f} | Acc {acc:.4f}")


# -------------------------
# 5. Eval
# -------------------------
@torch.no_grad()
def evaluate(epoch):
    model1.eval()
    model2.eval()

    correct = 0
    total = 0
    total_loss = 0.0

    for x, y in test_loader:
        x = x.to(DEVICE)
        y = y.to(DEVICE)

        h = model1(x)
        logits = model2(h)

        loss = F.cross_entropy(logits, y)

        total_loss += loss.item() * x.size(0)
        pred = logits.argmax(dim=-1)

        correct += (pred == y).sum().item()
        total += y.size(0)

    avg_loss = total_loss / total
    acc = correct / total

    print(f"[Eval]  Epoch {epoch} | Loss {avg_loss:.4f} | Acc {acc:.4f}")


# -------------------------
# 6. Run
# -------------------------
for epoch in range(1, EPOCHS + 1):
    train_one_epoch(epoch)
    evaluate(epoch)

Epoch 1 | Step 0 | Loss 2.3126 | Boundary grad norm 0.0204
Epoch 1 | Step 100 | Loss 0.5228 | Boundary grad norm 0.0311
Epoch 1 | Step 200 | Loss 0.2994 | Boundary grad norm 0.0250
Epoch 1 | Step 300 | Loss 0.4364 | Boundary grad norm 0.0316
Epoch 1 | Step 400 | Loss 0.2424 | Boundary grad norm 0.0266
[Train] Epoch 1 | Loss 0.4915 | Acc 0.8569
[Eval]  Epoch 1 | Loss 0.2412 | Acc 0.9302
Epoch 2 | Step 0 | Loss 0.1346 | Boundary grad norm 0.0177
Epoch 2 | Step 100 | Loss 0.2655 | Boundary grad norm 0.0254
Epoch 2 | Step 200 | Loss 0.2440 | Boundary grad norm 0.0291
Epoch 2 | Step 300 | Loss 0.1357 | Boundary grad norm 0.0251
Epoch 2 | Step 400 | Loss 0.1287 | Boundary grad norm 0.0231
[Train] Epoch 2 | Loss 0.2080 | Acc 0.9392
[Eval]  Epoch 2 | Loss 0.1590 | Acc 0.9533
Epoch 3 | Step 0 | Loss 0.1497 | Boundary grad norm 0.0268
Epoch 3 | Step 100 | Loss 0.1522 | Boundary grad norm 0.0265
Epoch 3 | Step 200 | Loss 0.1326 | Boundary grad norm 0.0261
Epoch 3 | Step 300 | Loss 0.1507 | Bounda